# NTPC PPE Detection PoC: Stage 2 PPE Detector Training

This notebook trains a YOLOv11n object detector on the cropped person dataset prepared in the previous notebook. The model will run locally on your RTX 4070 GPU.

In [1]:
# 1. Import dependencies and check GPU status
import torch
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory / (1024**3), "GB")

PyTorch version: 2.6.0+cu124
CUDA Available: True
GPU Device Name: NVIDIA GeForce RTX 4070 Laptop GPU
VRAM: 7.99560546875 GB


In [2]:
# 2. Initialize YOLOv11 model
# We start with pretrained yolov11n (nano) or yolov11s (small)
# yolov11n is faster (ideal for target FPS), yolov11s has slightly better representation capacity.
model = YOLO('../yolov11n.pt') # downloads pretrained weights automatically

In [ ]:
# 3. Configure and start fine-tuning
# Optimized training params for safety apparel on RTX 4070 laptop GPU:
results = model.train(
    data='../datasets/ppe_crops/data.yaml', 
    epochs=30,             # Sufficient for fine-tuning on pre-trained weights
    imgsz=320,             # Reduced from 640 (huge speed up for crops)
    batch=64,              # Increased from 16 to utilize RTX 4070 VRAM
    device=0,              # Local GPU index 0
    workers=8,             # Increased dataloading threads
    cache=True,            # Cache images in RAM to remove disk I/O bottlenecks
    val=False,             # Disable per-epoch validation to save time
    lr0=0.01,              # Initial learning rate
    optimizer='AdamW',     # Modern optimizer with better generalization
    save=True,             # Save checkpoints
    project='ntpc_ppe',    # Project folder
    name='stage2_crop_det'# Training run name
)

Ultralytics 8.4.66  Python-3.13.14 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../datasets/ppe_crops/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=../yolov11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=stage2_crop_det-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=A

: 

In [2]:
model = YOLO('../runs/detect/ntpc_ppe/stage2_crop_det-2/weights/last.pt')
model.train(resume=True, workers=2)

Ultralytics 8.4.66  Python-3.13.14 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\datasets\ppe_crops\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=..\runs\detect\ntpc_ppe\stage2_crop_det-2\weights\last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=stage2_crop_det-2, nbs=64, nms=Fal

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000020686623390>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

In [3]:
# 4. Validate Model & Print Metrics
metrics = model.val()
print("Validation Mean AP (mAP@50):", metrics.box.map50)
print("Validation Precision (per class):", metrics.box.p)
print("Validation Recall (per class):", metrics.box.r)

Ultralytics 8.4.66  Python-3.13.14 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 61.047.1 MB/s, size: 31.2 KB)
val: Scanning C:\Users\ashmi\OneDrive\Documents\ntpc_vocational\datasets\ppe_crops\val\labels.cache... 6524 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6524/6524 3.0Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 408/408 13.2it/s 30.9s.1sss
                   all       6524       9274      0.874      0.841      0.881      0.597
                helmet       4023       5078      0.939       0.86      0.914      0.633
                  head        637        701      0.836      0.786      0.849      0.557
                  vest       2516       3495      0.846      0.876      0.881      0.599
Speed: 0.1ms preprocess, 0.7ms inference, 0.0ms loss, 0.9

In [1]:
# 5. Save and Export model weights
import os
import shutil

models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

best_weights = os.path.join(model.trainer.save_dir, 'weights', 'best.pt')
target_weights = os.path.join(models_dir, 'ppe_crop_detector.pt')

if os.path.exists(best_weights):
    shutil.copy(best_weights, target_weights)
    print(f"Successfully saved fine-tuned model to: {target_weights}")
else:
    print("Error: best.pt weights not found. Check training runs path.")

NameError: name 'model' is not defined